# Week 3 build · An MLP that generalises, broken and fixed

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mjtri/learnml/blob/main/notebooks/week-03.ipynb)

**~3 hours · CPU only · run top to bottom.**

**The one rule:** every experiment is preceded by a 🔮 **predict cell**. Type what you expect and how sure you are *before* running the experiment. The notebook refuses empty predictions. A rough guess is fine; the gap between guess and result is what you are here to collect (canvas steps 7–8: *predict before running*, *log the surprise*).

| Part | Topic | Time |
|---|---|---|
| A | MNIST, a Dataset of your own, DataLoader shapes | 20 min |
| B | The canonical loop: first loss 2.30, then past 97 % | 30 min |
| C | Softmax and cross-entropy by hand | 20 min |
| D | Batch size; SGD, momentum, Adam | 25 min |
| E | Overfit 500 images on purpose, then regularize | 30 min |
| F | Break it three ways, fix it three ways | 45 min |
| G | Stretch: residual connections at depth | 15 min |

In [ ]:
#@title Setup: run me first
import numpy as np, torch, matplotlib.pyplot as plt
from IPython.display import Audio, Markdown, display
torch.manual_seed(0); np.random.seed(0)
_LEDGER = {}

def predict(tag, prediction, confidence):
    """Log a prediction BEFORE the experiment. Refuses empty predictions."""
    if not str(prediction).strip():
        raise ValueError(f"[{tag}] Write a prediction first. A rough guess is fine; an empty one is not.")
    _LEDGER[tag] = {"prediction": str(prediction).strip(), "confidence": int(confidence), "actual": "", "surprise": ""}
    print(f"Logged [{tag}] at {confidence}% confidence: {prediction}")

def record(tag, what_happened, surprise="none"):
    if tag not in _LEDGER:
        raise ValueError(f"[{tag}] has no prediction yet. Run its predict cell first.")
    _LEDGER[tag].update(actual=str(what_happened).strip(), surprise=surprise)
    print(f"Recorded [{tag}] surprise={surprise}")

def reveal():
    rows = ["| # | prediction | sure | what happened | surprise |", "|---|---|---|---|---|"]
    rows += [f"| {t} | {r['prediction']} | {r['confidence']}% | {r['actual']} | {r['surprise']} |" for t, r in _LEDGER.items()]
    print("\n".join(rows)); display(Markdown("\n".join(rows)))

print("torch", torch.__version__, "| ready")

In [ ]:
#@title Flags: SMOKE shrinks everything so the notebook runs on a laptop CPU in ~1 min. Set it to False for the real run.
import os, math, gzip, urllib.request
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
SMOKE = True           # True = tiny data/model/steps (plumbing check). False = the real run on Colab (~10 min CPU).
N_TRAIN, N_VAL = (3000, 1000) if SMOKE else (50000, 10000)
HID    = 64 if SMOKE else 256          # hidden width
EPOCHS = 1 if SMOKE else 6             # Part B
STEPS  = 150 if SMOKE else 300         # steps per experiment in Parts D, F, G
print("SMOKE" if SMOKE else "REAL", "run |", N_TRAIN, "train,", N_VAL, "val, hidden", HID)

---
## Part A · MNIST, a Dataset, a DataLoader (20 min)

**Words used in this part.** *MNIST*: 70,000 28×28 grey images of handwritten digits, ten classes. *Dataset*: a class that answers `len(ds)` and `ds[i]`. *DataLoader*: shuffles, fetches and stacks examples into batches. *uint8*: one byte per pixel, 0–255; models want floats near 0–1.

The raw files are the original IDX format (four gzip files, 11 MB). No `torchvision` needed: we parse the header ourselves, which is also a shape drill.

In [ ]:
DATA, BASE = "data/mnist", "https://ossci-datasets.s3.amazonaws.com/mnist/"
os.makedirs(DATA, exist_ok=True)
def load_idx(name):
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        urllib.request.urlretrieve(BASE + name, path)
    with gzip.open(path, "rb") as f:
        raw = f.read()
    ndim = raw[3]                                    # byte 3 of the magic number = number of axes
    shape = [int.from_bytes(raw[4 + 4*i: 8 + 4*i], "big") for i in range(ndim)]
    return torch.frombuffer(bytearray(raw[4 + 4*ndim:]), dtype=torch.uint8).reshape(shape)

X_all,  y_all  = load_idx("train-images-idx3-ubyte.gz"), load_idx("train-labels-idx1-ubyte.gz").long()
X_test, y_test = load_idx("t10k-images-idx3-ubyte.gz"),  load_idx("t10k-labels-idx1-ubyte.gz").long()
print("train", tuple(X_all.shape), X_all.dtype, "| labels", tuple(y_all.shape), "| test", tuple(X_test.shape))
print("pixel range", X_all.min().item(), "to", X_all.max().item(), "| first labels", y_all[:8].tolist())

In [ ]:
#@title 🔮 Predict A1: The Dataset returns (image/255 as float with shape (1,28,28), label). With batch size 64, what are the shapes and dtypes of x and y in one batch, and how many batches does one epoch of N_TRAIN examples take?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("A1", prediction, confidence)

In [ ]:
class Digits(Dataset):
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return self.X[i].float().div(255).unsqueeze(0), self.y[i]     # (1, 28, 28) float, scalar long

perm = torch.randperm(60000)                       # split by shuffled index: train and val never overlap
tr_idx, va_idx = perm[:N_TRAIN], perm[50000:50000 + N_VAL]
train_ds, val_ds = Digits(X_all[tr_idx], y_all[tr_idx]), Digits(X_all[va_idx], y_all[va_idx])
BATCH = 64
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=1000)      # evaluation: bigger batches, no shuffle
x, y = next(iter(train_dl))
print("x", tuple(x.shape), x.dtype, "| y", tuple(y.shape), y.dtype, "| batches per epoch", len(train_dl))
print("last batch holds", N_TRAIN % BATCH or BATCH, "examples")

In [ ]:
#@title ✍️ A1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("A1", what_happened, surprise)

In [ ]:
fig, ax = plt.subplots(1, 8, figsize=(10, 1.6))
for k in range(8):
    ax[k].imshow(x[k, 0], cmap="gray"); ax[k].set_title(int(y[k])); ax[k].axis("off")
plt.show()

---
## Part B · The canonical loop (30 min)

**Words used in this part.** *nn.Module*: owns parameters, defines `forward`; `model.parameters()` walks the tree. *optimizer*: reads every `.grad`, applies the update. *accuracy*: fraction of highest-score-is-right; reported, never trained on. *epoch*: one pass over the training set.

`make_mlp` is the only model factory in this notebook; every later part reuses it.

In [ ]:
def make_mlp(hidden=HID, depth=1, act=nn.ReLU, norm=None):
    """Flatten -> [Linear -> (norm) -> act] x depth -> Linear(10)."""
    layers, n_in = [nn.Flatten()], 784
    for _ in range(depth):
        layers.append(nn.Linear(n_in, hidden))
        if norm is not None:
            layers.append(norm(hidden))
        layers.append(act()); n_in = hidden
    layers.append(nn.Linear(n_in, 10))
    return nn.Sequential(*layers)

@torch.no_grad()
def accuracy(model, dl):
    model.eval(); correct = n = 0                 # eval(): dropout off, BatchNorm uses running averages
    for x, y in dl:
        correct += (model(x).argmax(1) == y).sum().item(); n += len(y)
    model.train(); return correct / n

model = make_mlp()
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
def train(model, opt, dl, epochs=1, max_steps=None):
    """The canonical loop. Returns the loss at every step; stops early if the loss is not finite."""
    losses = []
    for ep in range(epochs):
        for x, y in dl:
            loss = F.cross_entropy(model(x), y)     # forward + loss (takes logits, does its own softmax)
            opt.zero_grad()                         # forget the last batch
            loss.backward()                         # fill every .grad
            opt.step()                              # w <- w - lr * w.grad
            losses.append(loss.item())
            if not math.isfinite(losses[-1]) or (max_steps and len(losses) >= max_steps):
                return losses
    return losses

In [ ]:
#@title 🔮 Predict B1: Before any training, ten classes: what is the loss on one batch, and what is the validation accuracy? Give numbers.
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("B1", prediction, confidence)

In [ ]:
model = make_mlp()
x, y = next(iter(train_dl))
print(f"first loss  {F.cross_entropy(model(x), y).item():.3f}   (knowing nothing = -ln(1/10) = {math.log(10):.3f})")
print(f"val accuracy {accuracy(model, val_dl):.3f}   (guessing = 0.100)")

In [ ]:
#@title ✍️ B1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("B1", what_happened, surprise)

In [ ]:
#@title 🔮 Predict B2: SGD, lr 0.2, batch 64, EPOCHS epochs of the loop. Predict the final validation accuracy (real run: 6 epochs on 50,000 images; SMOKE: 1 epoch on 3,000).
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("B2", prediction, confidence)

In [ ]:
torch.manual_seed(0); model_b = make_mlp()
opt = torch.optim.SGD(model_b.parameters(), lr=0.2)
losses_b, val_acc = [], []
for ep in range(EPOCHS):
    losses_b += train(model_b, opt, train_dl)
    val_acc.append(accuracy(model_b, val_dl))
    print(f"epoch {ep+1}: train loss (last 50 steps) {np.mean(losses_b[-50:]):.3f}   val accuracy {val_acc[-1]:.4f}")
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(losses_b, lw=0.5); ax[0].set_yscale("log"); ax[0].set_title("loss per step (noisy: one batch each)")
ax[1].plot(range(1, EPOCHS + 1), val_acc, "o-"); ax[1].set_title("validation accuracy per epoch"); plt.show()
print("PASSED 97 %" if val_acc[-1] > 0.97 else "below 97 %" + (" (expected in SMOKE mode)" if SMOKE else ": train longer, or raise the learning rate a little"))

In [ ]:
#@title ✍️ B2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("B2", what_happened, surprise)

The per-step curve is noisy because every point is one batch of 64 (Part D). Read the trend, not the wiggles. The *validation* number is the one that matters (Part E).

---
## Part C · Softmax and cross-entropy by hand (20 min)

**Words used in this part.** *logits*: the ten raw scores per image. *softmax*: `exp(z) / exp(z).sum()`. *one-hot*: label as a vector with a single 1. *cross-entropy*: `-log p[true]`, averaged over the batch. *gradient at the logits*: `p - onehot`.

In [ ]:
#@title 🔮 Predict C1: Take the trained model's logits for one batch. (a) Will hand-written softmax + -log p[true] match F.cross_entropy to 1e-5? (b) Will the autograd gradient at the logits equal p - onehot? (c) If you feed softmax(logits) INTO F.cross_entropy by mistake, is the loss bigger or smaller than the correct one?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("C1", prediction, confidence)

In [ ]:
x, y = next(iter(val_dl))
logits = model_b(x).detach().requires_grad_(True)
p = torch.exp(logits) / torch.exp(logits).sum(1, keepdim=True)         # softmax, by hand
onehot = F.one_hot(y, 10).float()
loss_hand = -(torch.log(p) * onehot).sum(1).mean()                      # -log p[true], averaged
loss_lib = F.cross_entropy(logits, y)
print(f"by hand {loss_hand.item():.6f}   F.cross_entropy {loss_lib.item():.6f}   match: {abs(loss_hand - loss_lib).item() < 1e-5}")
loss_lib.backward()
print("gradient at logits == (p - onehot)/B ?", torch.allclose(logits.grad, (p - onehot).detach() / len(y), atol=1e-6))
double = F.cross_entropy(p.detach(), y)                                 # the double-softmax bug
print(f"double softmax loss {double.item():.3f}   (correct {loss_lib.item():.3f}; floor for 10 classes = ln(1 + 9/e) = {math.log(1 + 9/math.e):.3f})")

In [ ]:
#@title ✍️ C1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("C1", what_happened, surprise)

Probabilities live in [0, 1], so after a second softmax the biggest possible gap between "logits" is 1, and the loss can never drop below **1.46** for ten classes. If a loss curve plateaus there, you know the bug without reading the code.

---
## Part D · Batch size; SGD, momentum, Adam (25 min)

**Words used in this part.** *mini-batch*: a random handful of examples per step; its gradient is a noisy estimate of the full one, spread ∝ 1/√B. *SGD*: step = −lr × gradient. *momentum*: step along a running average of gradients (memory 0.9 ≈ ten steps). *Adam*: running averages of gradient and gradient²; each parameter moves ≈ lr per step.

Every run below starts from the **same initial weights** (`torch.manual_seed(0)`) and gets the **same number of steps**, so only the thing we change differs.

In [ ]:
def fresh(**kw):
    torch.manual_seed(0); return make_mlp(**kw)

def smooth(v, k=20):
    v = np.asarray(v, dtype=float); return np.convolve(v, np.ones(k) / k, mode="valid") if len(v) > k else v

def compare(runs, title):
    """runs: {label: losses}. Plots smoothed curves and prints the mean of the last 20 losses."""
    for label, losses in runs.items():
        plt.plot(smooth(losses), label=label)
        print(f"{label:<28} last-20 mean loss {np.mean(losses[-20:]):.3f}   steps {len(losses)}")
    plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss (smoothed)"); plt.title(title); plt.legend(); plt.show()

In [ ]:
#@title 🔮 Predict D1: SGD lr 0.1, STEPS steps each, batch size 8 vs 64 vs 512. After the same number of steps: which has the lowest loss, which the noisiest curve, and which used the most compute (steps × batch)?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("D1", prediction, confidence)

In [ ]:
runs = {}
for B in [8, 64, 512]:
    dl = DataLoader(train_ds, batch_size=B, shuffle=True)
    m = fresh(); opt = torch.optim.SGD(m.parameters(), lr=0.1)
    steps = STEPS if B < 512 else max(STEPS // 4, 40)     # 512-batches cost 8x per step; keep the cell quick
    runs[f"batch {B} ({steps * B:,} images seen)"] = train(m, opt, dl, epochs=100, max_steps=steps)
compare(runs, "same steps, different batch size")

In [ ]:
#@title ✍️ D1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("D1", what_happened, surprise)

In [ ]:
#@title 🔮 Predict D2: Batch 64, STEPS steps, same start: SGD lr 0.01 vs SGD lr 0.01 + momentum 0.9 vs Adam lr 0.001. Rank them by final loss. Which two do you expect to be close?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("D2", prediction, confidence)

In [ ]:
runs = {}
for name, make_opt in {
    "SGD lr 0.01":              lambda ps: torch.optim.SGD(ps, lr=0.01),
    "SGD lr 0.01 momentum 0.9": lambda ps: torch.optim.SGD(ps, lr=0.01, momentum=0.9),
    "Adam lr 0.001":            lambda ps: torch.optim.Adam(ps, lr=0.001),
}.items():
    m = fresh(); runs[name] = train(m, make_opt(m.parameters()), train_dl, epochs=100, max_steps=STEPS)
compare(runs, "same steps, different optimizer")

In [ ]:
#@title ✍️ D2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("D2", what_happened, surprise)

Momentum with memory 0.9 behaves like SGD with roughly 10× the learning rate once the gradients agree, which is why it caught up. Adam moves every parameter about `lr` per step regardless of gradient size: its learning rate is a *distance*, SGD's is a *gain*. Week 1's millimetre bug would not have hurt Adam.

---
## Part E · Overfit on purpose, then regularize (30 min)

**Words used in this part.** *training / validation split*: fit on one, measure on the other; the model never sees validation labels. *overfitting*: training loss keeps falling while validation loss rises. *weight decay*: shrink weights towards zero each step (AdamW). *dropout*: zero a random fraction of a layer's outputs during training only. *early stopping*: keep the parameters from the best validation epoch.

A model with 200,000 parameters and only 500 images is the fastest way to *see* overfitting.

In [ ]:
N_SMALL, EP_SMALL = (200, 10) if SMOKE else (500, 30)
small_dl = DataLoader(Digits(X_all[tr_idx[:N_SMALL]], y_all[tr_idx[:N_SMALL]]), batch_size=32, shuffle=True)

def run_small(model, opt, epochs=EP_SMALL):
    """Train on the tiny set; return per-epoch (train acc, val acc, val loss) and the best-val-epoch accuracy."""
    hist = []
    for ep in range(epochs):
        train(model, opt, small_dl)
        model.eval()
        with torch.no_grad():
            xv, yv = next(iter(val_dl)); vloss = F.cross_entropy(model(xv), yv).item()
        model.train()
        hist.append((accuracy(model, small_dl), accuracy(model, val_dl), vloss))
    h = np.array(hist); return h, h[h[:, 2].argmin(), 1]        # early stopping = pick the min-val-loss epoch

In [ ]:
#@title 🔮 Predict E1: Train on N_SMALL images for EP_SMALL epochs with Adam lr 0.001. Predict the final TRAIN accuracy and the final VALIDATION accuracy, and whether the validation loss goes up at some point.
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("E1", prediction, confidence)

In [ ]:
m = fresh(); h, best = run_small(m, torch.optim.Adam(m.parameters(), lr=0.001))
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(h[:, 0], label="train acc"); ax[0].plot(h[:, 1], label="val acc"); ax[0].legend(); ax[0].set_xlabel("epoch")
ax[1].plot(h[:, 2]); ax[1].set_title("validation loss"); ax[1].set_xlabel("epoch"); plt.show()
print(f"final: train acc {h[-1, 0]:.3f}   val acc {h[-1, 1]:.3f}   gap {h[-1, 0] - h[-1, 1]:.3f}")
print(f"validation loss: lowest at epoch {h[:, 2].argmin() + 1}, then {'rises' if h[-1, 2] > h[:, 2].min() * 1.05 else 'stays flat'}")

In [ ]:
#@title ✍️ E1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("E1", what_happened, surprise)

Look at the two right-hand curves together: the validation **loss** climbs after its dip while validation **accuracy** stays flat. The model is not getting more answers wrong; it is getting more *confident* about the ones it already gets wrong, and cross-entropy charges for confidence. That is why early stopping watches the loss, and why a rising validation loss is the first sign of overfitting even when accuracy has not moved.

In [ ]:
#@title 🔮 Predict E2: Same tiny set, three fixes: AdamW weight decay; dropout after the hidden layer; early stopping (best-val-loss epoch of the plain run). Rank the three by validation accuracy, and say how much the best one gains over plain: about 1 point, 5, or 10? Does any fix close the gap?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("E2", prediction, confidence)

In [ ]:
results = {"plain (last epoch)": h[-1, 1], "early stopping (best epoch)": best}
m = fresh(); results["weight decay 0.1 (AdamW)"] = run_small(m, torch.optim.AdamW(m.parameters(), lr=0.001, weight_decay=0.1))[0][-1, 1]
torch.manual_seed(0)
m = nn.Sequential(nn.Flatten(), nn.Linear(784, HID), nn.ReLU(), nn.Dropout(0.5), nn.Linear(HID, 10))
results["dropout 0.5"] = run_small(m, torch.optim.Adam(m.parameters(), lr=0.001))[0][-1, 1]
for k, v in sorted(results.items(), key=lambda kv: -kv[1]):
    print(f"{k:<28} val accuracy {v:.3f}")
print("for scale: Part B's model, trained on", N_TRAIN, "images, reached", f"{val_acc[-1]:.3f}")

In [ ]:
#@title ✍️ E2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("E2", what_happened, surprise)

The gains are small, a point or two, and no regularizer beats *more data*: Part B's number is out of reach for any of these. Regularizers earn their keep on bigger models trained longer; on 500 images the honest fix is 5,000 images. Karpathy's other check, **overfit one batch**, is the mirror image: a model that *cannot* drive the loss to zero on two images has a bug.

In [ ]:
m = fresh(); opt = torch.optim.Adam(m.parameters(), lr=0.01)
x2, y2 = x[:2], y[:2]                                       # two images
for step in range(200):
    loss = F.cross_entropy(m(x2), y2); opt.zero_grad(); loss.backward(); opt.step()
print(f"loss on two images after 200 steps: {loss.item():.2e}   (pipeline OK if this is ~0)")

---
## Part F · Break it three ways, fix it three ways (45 min)

**Words used in this part.** *initialization*: the weights' starting scale; a unit summing n inputs needs weights of spread ≈ 1/√n. *Kaiming init*: spread √(2/n) for ReLU. *BatchNorm*: normalize each unit across the batch, then learn a scale and shift. *vanishing / exploding gradient*: the signal shrinks or grows by a constant factor per layer.

Three broken nets, each with one thing wrong:

- **A**: a fine 2-hidden-layer tanh MLP, but SGD with **lr = 10**.
- **B**: a 2-hidden-layer ReLU MLP, but every weight drawn with spread **1.0** instead of ≈ 1/√n (30× too big for 784 inputs).
- **C**: **12 tanh layers** with PyTorch's default init (spread 1/√(3n), too small for tanh at depth).

Three fixes: **lr ÷ 50**, **Kaiming init**, **BatchNorm after every Linear**. First the fingerprints, then the 3 × 3 table.

In [ ]:
def init_normal(model, std):
    for m in model.modules():
        if isinstance(m, nn.Linear): nn.init.normal_(m.weight, 0, std); nn.init.zeros_(m.bias)
    return model

def kaiming(model):
    for m in model.modules():
        if isinstance(m, nn.Linear): nn.init.kaiming_normal_(m.weight, nonlinearity="relu"); nn.init.zeros_(m.bias)
    return model

BROKEN = {   # name: (learning rate, model builder taking norm=...)
    "A: lr = 10 (tanh)":  (10.0, lambda norm: fresh(depth=2, act=nn.Tanh, norm=norm)),
    "B: weights spread 1": (0.1, lambda norm: init_normal(fresh(depth=2, norm=norm), 1.0)),
    "C: 12 tanh layers":  (0.1, lambda norm: fresh(depth=12, act=nn.Tanh, norm=norm)),
}

def run_case(name, fix="none", steps=STEPS):
    lr, build = BROKEN[name]
    m = build(nn.BatchNorm1d if fix == "BatchNorm" else None)
    if fix == "Kaiming init": kaiming(m)
    if fix == "lr / 50": lr = lr / 50
    first = F.cross_entropy(m(x), y).item()
    losses = train(m, torch.optim.SGD(m.parameters(), lr=lr), train_dl, epochs=100, max_steps=steps)
    return first, losses[-1], len(losses)

In [ ]:
#@title 🔮 Predict F1: Fingerprints. For each of A, B, C predict the FIRST loss (≈2.3 or far above 2.3) and what the loss does over STEPS steps (falls, stays stuck near 2.3, or explodes into the hundreds / NaN).
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("F1", prediction, confidence)

In [ ]:
for name in BROKEN:
    first, last, n = run_case(name)
    print(f"{name:<22} first loss {first:9.3f}   after {n:3d} steps: {last if math.isfinite(last) else 'NaN (diverged)'}")

In [ ]:
#@title ✍️ F1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("F1", what_happened, surprise)

In [ ]:
#@title 🔮 Predict F2: The 3 x 3 table. For each broken net, which single fix rescues it (loss well below 2.3 after STEPS steps): lr/50, Kaiming init, BatchNorm? Write your predicted table, e.g. 'A: only lr; B: init and BN; C: ...'
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("F2", prediction, confidence)

In [ ]:
FIXES = ["none", "lr / 50", "Kaiming init", "BatchNorm"]
print(f"{'':<22}" + "".join(f"{f:>14}" for f in FIXES))
for name in BROKEN:
    row = []
    for fix in FIXES:
        first, last, n = run_case(name, fix)
        row.append("NaN" if not math.isfinite(last) else f"{last:.2f}" + (" ok" if last < 1.0 else ""))
    print(f"{name:<22}" + "".join(f"{r:>14}" for r in row))
print("ok = loss below 1.0 after", STEPS, "steps (2.30 = still knows nothing)")

In [ ]:
#@title ✍️ F2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("F2", what_happened, surprise)

**Reading the table.** A learning-rate fingerprint (loss explodes within a few steps) is fixed only by the learning rate; init and normalization cannot save a step that is too long. An init fingerprint (first loss far above 2.3) is fixed by re-initializing, and helped by normalizing, because normalization resets the spread whatever the weights (the last layer has no norm, so it is slower). A depth fingerprint (stuck at 2.3) needs the signal to survive the layers: init or normalization, never a smaller learning rate. One trap: the same too-large learning rate on a **ReLU** net does not explode; it kills every unit and the loss sits at 2.30, wearing the depth fingerprint. When in doubt, halve the learning rate first; it is the cheapest experiment. This is the `core-optim` done-when: name the fix before you try it.

---
## Part G · Stretch: residual connections (15 min)

**Words used in this part.** *residual connection*: each block computes `x + f(x)`; the gradient has an identity path back through every block. *plain*: `f(x)` only.

Same depth, same init, same steps; the only difference is one `+ x`. Watch the loss **and** the gradient reaching the first layer.

In [ ]:
class Block(nn.Module):
    def __init__(self, h, residual):
        super().__init__(); self.lin, self.residual = nn.Linear(h, h), residual
    def forward(self, x):
        out = torch.tanh(self.lin(x))
        return x + out if self.residual else out

class DeepMLP(nn.Module):
    def __init__(self, depth, residual, h=HID):
        super().__init__()
        self.inp = nn.Linear(784, h); self.out = nn.Linear(h, 10)
        self.blocks = nn.Sequential(*[Block(h, residual) for _ in range(depth)])
    def forward(self, x):
        return self.out(self.blocks(torch.tanh(self.inp(x.flatten(1)))))

DEPTH = 12 if SMOKE else 24
def first_layer_grad(model):
    model.zero_grad(); F.cross_entropy(model(x), y).backward()
    return model.inp.weight.grad.norm().item()

In [ ]:
#@title 🔮 Predict G1: DEPTH tanh blocks, default init, SGD lr 0.03, STEPS steps: plain vs residual. Which trains? Predict the ratio of the gradient norm at the first layer, residual / plain, at the start (1x, 10x, 1000x?).
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("G1", prediction, confidence)

In [ ]:
runs = {}
for residual in [False, True]:
    torch.manual_seed(0); m = DeepMLP(DEPTH, residual)
    g = first_layer_grad(m)
    runs[f"{'residual' if residual else 'plain'} (first-layer grad {g:.1e})"] = train(
        m, torch.optim.SGD(m.parameters(), lr=0.03), train_dl, epochs=100, max_steps=STEPS)
compare(runs, f"{DEPTH} tanh blocks: plain vs residual")

In [ ]:
#@title ✍️ G1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("G1", what_happened, surprise)

The plain stack multiplies the gradient by a factor below one at every block; the residual stack adds an identity term, so the product never collapses. In week 5 you will see this exact wire drawn twice inside every transformer block.

---
## Wrap up

The test set has not been touched. Look at it **once**, with Part B's model, and that is the number you would report:

In [ ]:
test_dl = DataLoader(Digits(X_test, y_test), batch_size=1000)
print(f"Part B model: validation {val_acc[-1]:.4f}   test (looked at once) {accuracy(model_b, test_dl):.4f}")

Run the cell below, copy the table into your insight ledger next to `core-optim`, and mark the biggest surprise.
Then on the laptop:

```
python track.py done week-03/build --rating <1-5> --minutes 180 --note "biggest surprise"
python gen_week.py 4
```

In [ ]:
reveal()